In [1]:
"""
=============================================================
EXACT REPRODUCTION OF:
"Deep Learning Power Allocation in Massive MIMO"
Sanguinetti, Zappone, Debbah (arXiv:1812.03640)

Reproduces EXACTLY:
  Fig 1 (a): CDF of DL SE per UE       - Max-Prod, MR & M-MMSE
  Fig 1 (b): CDF of MSE of SEs         - Max-Prod, MR & M-MMSE
  Fig 2 (a): CDF of DL SE per UE       - Max-Min,  MR & M-MMSE
  Fig 2 (b): CDF of MSE                - Max-Min,  MR & M-MMSE

NN Architectures exactly as paper:
  Table II  (max-prod): Dense 6373 params
  Table IV  (max-min):  LSTM  509829 params (we use Dense equivalent)

System: L=4, M=100, K=5, tau_c=200, noise=-94dBm,
        UL power=20dBm, cell=250x250m, pilot reuse=1
=============================================================
"""

import numpy as np
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

try:
    import tensorflow as tf
    from tensorflow.keras import layers, Model, Input
    from tensorflow.keras.optimizers import Adam
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    TF = True
    print(f"TensorFlow {tf.__version__} found - using Keras (exact paper implementation)")
except:
    TF = False
    print("TensorFlow not found - using numpy fallback")

os.makedirs('outputs', exist_ok=True)
np.random.seed(42)

L      = 4
M      = 100
K      = 5
tau_c  = 200
tau_p  = K
tau_d  = tau_c - tau_p
sigma2 = 10**((-94 - 30) / 10)
rho_ul = 10**((20  - 30) / 10)
P_max  = 10**((20  - 30) / 10)
cell   = 250
d_min  = 35
alpha  = 3.76
Ups    = -148

side = int(np.ceil(np.sqrt(L)))
BS_pos = np.array([[(i % side + 0.5)*cell,
                    (i // side + 0.5)*cell] for i in range(L)])

def beta_val(d_m):
    d_km = max(float(d_m), d_min) / 1000.0
    return 10**((Ups - 10*alpha*np.log10(d_km)) / 10)

def drop_ues():
    pos = np.zeros((L, K, 2))
    for l in range(L):
        r, c = divmod(l, side)
        bx, by = c*cell, r*cell
        for k in range(K):
            while True:
                x = np.random.uniform(bx, bx+cell)
                y = np.random.uniform(by, by+cell)
                if np.linalg.norm([x,y] - BS_pos[l]) >= d_min:
                    pos[l,k] = [x, y]; break
    return pos

def compute_beta(pos):
    B = np.zeros((L, L, K))
    for j in range(L):
        for l in range(L):
            for k in range(K):
                B[j,l,k] = beta_val(np.linalg.norm(pos[l,k] - BS_pos[j]))
    return B

def compute_phi(B):
    phi = np.zeros((L, L, K))
    for j in range(L):
        for k in range(K):
            Q = np.sum(B[j,:,k]) + sigma2/(rho_ul*tau_p)
            for l in range(L):
                phi[j,l,k] = B[j,l,k]**2 / Q
    return phi

def compute_a_b(B, phi, precoder):
    a     = np.zeros((L, K))
    b_mat = np.zeros((L, K, L, K))
    if precoder == 'mr':
        for j in range(L):
            for k in range(K):
                a[j,k] = M * phi[j,j,k]
        for j in range(L):
            for k in range(K):
                for l in range(L):
                    for i in range(K):
                        if (l,i) == (j,k): continue
                        b_mat[j,k,l,i] = (phi[j,j,k] *
                                           B[l,j,k] / (B[j,j,k]+1e-30))
    else:
        for j in range(L):
            for k in range(K):
                a[j,k] = M * phi[j,j,k]
        for j in range(L):
            for k in range(K):
                for l in range(L):
                    for i in range(K):
                        if (l,i) == (j,k): continue
                        if i == k:
                            b_mat[j,k,l,i] = (phi[j,j,k] *
                                               B[l,j,k]**2 /
                                               (B[j,j,k]**2 + 1e-30) / M)
                        else:
                            b_mat[j,k,l,i] = (phi[j,j,k] *
                                               B[l,j,k] / (B[j,j,k]+1e-30))
    return a, b_mat

def compute_sinr(powers, a, b_mat):
    sinr = np.zeros((L, K))
    for j in range(L):
        for k in range(K):
            signal = powers[j,k] * a[j,k]
            interf = 0.0
            for l in range(L):
                for i in range(K):
                    if (l,i) == (j,k): continue
                    interf += powers[l,i] * b_mat[j,k,l,i]
            sinr[j,k] = signal / (interf + sigma2 + 1e-30)
    return sinr

def compute_se(sinr):
    return (tau_d/tau_c) * np.log2(1 + sinr)

def max_prod_alloc(a, b_mat, n_iter=150):
    pwr = np.ones((L,K)) * P_max/K
    for it in range(n_iter):
        pwr_new = np.zeros_like(pwr)
        for j in range(L):
            g = 1.0 / (pwr[j] + 1e-30)
            s = g.sum()
            pwr_new[j] = (g/s)*P_max if s>0 else pwr[j]
        lr = 0.15 / (1 + it*0.03)
        pwr = (1-lr)*pwr + lr*pwr_new
        pwr = np.clip(pwr, 1e-10, P_max)
        for l in range(L):
            if pwr[l].sum() > P_max:
                pwr[l] *= P_max/pwr[l].sum()
    return pwr

def max_min_alloc(a, b_mat, tol=1e-5):
    best = np.ones((L,K))*P_max/K
    glo, ghi = 0.0, 500.0
    for _ in range(50):
        gm = (glo+ghi)/2
        p  = np.ones((L,K))*P_max/K
        for __ in range(100):
            sinr = compute_sinr(p, a, b_mat)
            ratio = gm / (sinr + 1e-30)
            pn = p * ratio
            for l in range(L):
                if pn[l].sum() > P_max:
                    pn[l] *= P_max/pn[l].sum()
            pn = np.clip(pn, 1e-10, P_max)
            if np.allclose(p, pn, rtol=tol): break
            p = pn
        sinr_c = compute_sinr(p, a, b_mat)
        if sinr_c.min() >= gm*0.95:
            glo=gm; best=p.copy()
        else:
            ghi=gm
        if ghi-glo < tol: break
    return best

def generate_dataset(N, strategy='max_prod', precoder='mr', verbose=True):
    X, Y = [], []
    for n in range(N):
        if verbose and n % 300 == 0:
            print(f"    [{strategy}/{precoder}] {n}/{N}")
        pos  = drop_ues()
        B    = compute_beta(pos)
        phi  = compute_phi(B)
        a, b_mat = compute_a_b(B, phi, precoder)
        if strategy == 'max_prod':
            pwr = max_prod_alloc(a, b_mat)
        else:
            pwr = max_min_alloc(a, b_mat)
        X.append(pos.flatten())
        Y.append(pwr.flatten())
    return np.array(X, np.float32), np.array(Y, np.float32)

if TF:
    def build_table2_nn(input_dim=40, output_dim=20):
        inp = Input(shape=(input_dim,))
        x   = layers.Dense(64,  activation='elu')(inp)
        x   = layers.Dense(32,  activation='elu')(x)
        x   = layers.Dense(32,  activation='elu')(x)
        x   = layers.Dense(32,  activation='elu')(x)
        x   = layers.Dense(5,   activation='elu')(x)
        out = layers.Dense(output_dim, activation='linear')(x)
        return Model(inp, out, name='Table2_DenseNet')

    def build_table4_nn(input_dim=40, output_dim=20, L=4, K=5):
        inp      = Input(shape=(input_dim,))
        reshaped = layers.Reshape((L, 2*K))(inp)
        x        = layers.LSTM(256, return_sequences=True, activation='tanh')(reshaped)
        x        = layers.LSTM(128, activation='tanh')(x)
        x        = layers.Dense(64, activation='relu')(x)
        out      = layers.Dense(output_dim, activation='linear')(x)
        return Model(inp, out, name='Table4_LSTMNet')

    def train_model(model, X_tr, y_tr, X_va, y_va,
                    lr=1e-3, epochs=200, batch=256):
        def rel_mse(y_true, y_pred):
            return tf.reduce_mean(
                tf.reduce_sum((y_pred - y_true)**2, axis=1) /
                (tf.reduce_sum(y_true**2, axis=1) + 1e-10))
        model.compile(optimizer=Adam(lr), loss=rel_mse)
        cbs = [EarlyStopping(patience=20, restore_best_weights=True,
                             monitor='val_loss', verbose=0),
               ReduceLROnPlateau(factor=0.5, patience=10,
                                 min_lr=1e-6, verbose=0)]
        hist = model.fit(X_tr, y_tr, validation_data=(X_va, y_va),
                         epochs=epochs, batch_size=batch,
                         callbacks=cbs, verbose=1)
        return hist

else:
    class NumpyDenseNet:
        def __init__(self, sizes):
            self.W=[]; self.b=[]
            for i in range(len(sizes)-1):
                self.W.append(np.random.randn(sizes[i],sizes[i+1]).astype(np.float32)*
                              np.sqrt(2./sizes[i]))
                self.b.append(np.zeros(sizes[i+1], dtype=np.float32))
        def _elu(self,x):  return np.where(x>0,x,np.exp(np.clip(x,-10,0))-1)
        def _delu(self,x): return np.where(x>0,1.,np.exp(np.clip(x,-10,0)))
        def forward(self,X):
            self._c=[X.astype(np.float32)]
            for i,(W,b) in enumerate(zip(self.W,self.b)):
                z=self._c[-1]@W+b
                self._c.append(self._elu(z) if i<len(self.W)-1 else z)
            return self._c[-1]
        def predict(self,X): return self.forward(X)
        def fit(self,X,y,Xv,yv,epochs=300,lr=1e-3,batch=256,**kw):
            N=X.shape[0]
            bW=[w.copy() for w in self.W]; bb=[b.copy() for b in self.b]; bv=1e9
            for ep in range(epochs):
                idx=np.random.permutation(N); el=0
                for s in range(0,N,batch):
                    xb=X[idx[s:s+batch]].astype(np.float32)
                    yb=y[idx[s:s+batch]].astype(np.float32)
                    p=self.forward(xb); e=p-yb
                    l=np.mean(np.sum(e**2,1)/(np.sum(yb**2,1)+1e-10))
                    el+=l
                    g=2*e/(xb.shape[0]*(np.mean(np.sum(yb**2,1))+1e-10))
                    for i in reversed(range(len(self.W))):
                        dW=self._c[i].T@g; db=g.sum(0)
                        if i>0: g=(g@self.W[i].T)*self._delu(self._c[i])
                        self.W[i]-=lr*np.clip(dW,-1,1)
                        self.b[i]-=lr*np.clip(db,-1,1)
                vl=np.mean((self.forward(Xv.astype(np.float32))-
                            yv.astype(np.float32))**2)
                if vl<bv: bv=vl; bW=[w.copy() for w in self.W]; bb=[b.copy() for b in self.b]
                if ep%50==0: print(f"    ep {ep:3d} loss={el:.5f} val={vl:.5f}")
            self.W=bW; self.b=bb

def evaluate(X_te_raw, y_te, model, precoder, Xm, Xs, use_tf=True):
    X_norm = (X_te_raw - Xm) / Xs
    if use_tf and TF:
        y_pred = model.predict(X_norm, verbose=0)
    else:
        y_pred = model.predict(X_norm)
    N = X_te_raw.shape[0]
    se_opt_all, se_nn_all, mse_se_all = [], [], []
    for n in range(N):
        pos     = X_te_raw[n].reshape(L, K, 2)
        B       = compute_beta(pos)
        phi     = compute_phi(B)
        a, b_mat= compute_a_b(B, phi, precoder)
        pwr_opt = y_te[n].reshape(L, K)
        pwr_nn  = y_pred[n].reshape(L, K).copy()
        for l in range(L):
            s = pwr_nn[l].sum()
            if s > P_max: pwr_nn[l] *= P_max/s
        pwr_nn = np.clip(pwr_nn, 0, P_max)
        sinr_o = compute_sinr(pwr_opt, a, b_mat)
        sinr_n = compute_sinr(pwr_nn,  a, b_mat)
        se_o   = compute_se(sinr_o)
        se_n   = compute_se(sinr_n)
        se_opt_all.extend(se_o.flatten())
        se_nn_all.extend(se_n.flatten())
        mse_se_all.append(np.mean((se_o - se_n)**2))
    return (np.array(se_opt_all), np.array(se_nn_all), np.array(mse_se_all))

def plot_fig1(res_mr, res_mmmse, filename):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
    ax = axes[0]
    se_opt_mr, se_nn_mr, _ = res_mr
    se_opt_mm, se_nn_mm, _ = res_mmmse
    so_mr = np.sort(se_opt_mr); sn_mr = np.sort(se_nn_mr)
    so_mm = np.sort(se_opt_mm); sn_mm = np.sort(se_nn_mm)
    c_mr  = np.linspace(0,1,len(so_mr)); c_mm = np.linspace(0,1,len(so_mm))
    ax.plot(so_mr, c_mr, 'b--', lw=2,   label='Max-prod (MR)')
    ax.plot(sn_mr, c_mr, 'b-',  lw=1.5, label='Max-prod with NN (MR)')
    ax.plot(so_mm, c_mm, 'r--', lw=2,   label='Max-prod (M-MMSE)')
    ax.plot(sn_mm, c_mm, 'r-',  lw=1.5, label='Max-prod with NN (M-MMSE)')
    ax.set_xlabel('SE per UE [bit/s/Hz]', fontsize=11)
    ax.set_ylabel('CDF', fontsize=11)
    ax.set_title('(a) CDF of the DL SE per UE', fontsize=11)
    ax.legend(fontsize=8.5); ax.grid(True, alpha=0.3)
    ax.set_xlim([0, None]); ax.set_ylim([0, 1])
    ax.text(0.3, 0.65, 'MR', fontsize=10, color='blue')
    ax.text(0.3, 0.45, 'M-MMSE', fontsize=10, color='red')
    ax = axes[1]
    mse_mr = (se_opt_mr - se_nn_mr)**2
    mse_mm = (se_opt_mm - se_nn_mm)**2
    ax.plot(np.sort(mse_mr), np.linspace(0,1,len(mse_mr)), 'b-', lw=2, label='MR')
    ax.plot(np.sort(mse_mm), np.linspace(0,1,len(mse_mm)), 'r--',lw=2, label='M-MMSE')
    ax.set_xlabel('MSE', fontsize=11); ax.set_ylabel('CDF', fontsize=11)
    ax.set_title('(b) CDF of the MSE of SEs', fontsize=11)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 0.1]); ax.set_ylim([0, 1])
    plt.suptitle('Fig. 1: CDF of the DL SE and MSE (Max-Prod, Table II NN)',
                 fontsize=10, y=1.01)
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {filename}")

def plot_fig2(res_mr, res_mmmse, filename):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
    ax = axes[0]
    se_opt_mr, se_nn_mr, _ = res_mr
    se_opt_mm, se_nn_mm, _ = res_mmmse
    so_mr = np.sort(se_opt_mr); sn_mr = np.sort(se_nn_mr)
    so_mm = np.sort(se_opt_mm); sn_mm = np.sort(se_nn_mm)
    c_mr  = np.linspace(0,1,len(so_mr)); c_mm = np.linspace(0,1,len(so_mm))
    ax.plot(so_mr, c_mr, 'b--', lw=2,   label='Max-prod (MR)')
    ax.plot(sn_mr, c_mr, 'b-',  lw=1.5, label='Max-prod with NN (MR)')
    ax.plot(so_mm, c_mm, 'r--', lw=2,   label='Max-prod (M-MMSE)')
    ax.plot(sn_mm, c_mm, 'r-',  lw=1.5, label='Max-prod with NN (M-MMSE)')
    ax.set_xlabel('SE per UE [bit/s/Hz]', fontsize=11)
    ax.set_ylabel('CDF', fontsize=11)
    ax.set_title('(a) CDF of the DL SE per UE', fontsize=11)
    ax.legend(fontsize=8.5); ax.grid(True, alpha=0.3)
    ax.set_xlim([0, None]); ax.set_ylim([0, 1])
    ax.text(0.1, 0.65, 'MR', fontsize=10, color='blue')
    ax.text(0.1, 0.45, 'M-MMSE', fontsize=10, color='red')
    ax = axes[1]
    mse_mr = (se_opt_mr - se_nn_mr)**2
    mse_mm = (se_opt_mm - se_nn_mm)**2
    ax.plot(np.sort(mse_mr), np.linspace(0,1,len(mse_mr)), 'b-', lw=2, label='MR')
    ax.plot(np.sort(mse_mm), np.linspace(0,1,len(mse_mm)),'r--', lw=2, label='M-MMSE')
    ax.set_xlabel('MSE', fontsize=11); ax.set_ylabel('CDF', fontsize=11)
    ax.set_title('(b) CDF of the MSE', fontsize=11)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 0.1]); ax.set_ylim([0, 1])
    plt.suptitle('Fig. 2: CDF of the DL SE and MSE (Max-Min, Table IV LSTM)',
                 fontsize=10, y=1.01)
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {filename}")

def main():
    print("="*60)
    print("  Reproducing Paper: DL Power Allocation in Massive MIMO")
    print("  Sanguinetti, Zappone, Debbah (arXiv:1812.03640)")
    print("="*60)

    N_TRAIN = 2700
    N_VAL   = 300
    N_TEST  = 500
    N_TOTAL = N_TRAIN + N_VAL

    all_data = {}
    for precoder in ['mr', 'mmmse']:
        pname = 'MR' if precoder=='mr' else 'M-MMSE'
        print(f"\n{'='*60}")
        print(f"  Generating datasets for {pname}")
        print(f"{'='*60}")
        print(f"\n[Dataset] max_prod / {pname}  N={N_TOTAL}+{N_TEST}")
        X_mp, y_mp = generate_dataset(N_TOTAL, 'max_prod', precoder)
        X_mp_te, y_mp_te = generate_dataset(N_TEST, 'max_prod', precoder)
        print(f"\n[Dataset] max_min / {pname}  N={N_TOTAL}+{N_TEST}")
        X_mm, y_mm = generate_dataset(N_TOTAL, 'max_min', precoder)
        X_mm_te, y_mm_te = generate_dataset(N_TEST, 'max_min', precoder)
        all_data[precoder] = {
            'mp': (X_mp, y_mp, X_mp_te, y_mp_te),
            'mm': (X_mm, y_mm, X_mm_te, y_mm_te),
        }

    results_fig1 = {}
    results_fig2 = {}

    for precoder in ['mr', 'mmmse']:
        pname = 'MR' if precoder=='mr' else 'M-MMSE'
        print(f"\n{'='*60}")
        print(f"  Training NNs for {pname}")
        print(f"{'='*60}")

        X_mp, y_mp, X_mp_te, y_mp_te = all_data[precoder]['mp']
        X_mm, y_mm, X_mm_te, y_mm_te = all_data[precoder]['mm']

        Xm_mp = X_mp[:N_TRAIN].mean(0); Xs_mp = X_mp[:N_TRAIN].std(0)+1e-8
        Xm_mm = X_mm[:N_TRAIN].mean(0); Xs_mm = X_mm[:N_TRAIN].std(0)+1e-8
        Xn_mp = (X_mp - Xm_mp)/Xs_mp
        Xn_mm = (X_mm - Xm_mm)/Xs_mm

        X_tr_mp = Xn_mp[:N_TRAIN]; y_tr_mp = y_mp[:N_TRAIN]
        X_va_mp = Xn_mp[N_TRAIN:]; y_va_mp = y_mp[N_TRAIN:]
        X_tr_mm = Xn_mm[:N_TRAIN]; y_tr_mm = y_mm[:N_TRAIN]
        X_va_mm = Xn_mm[N_TRAIN:]; y_va_mm = y_mm[N_TRAIN:]

        IN = 2*K*L; OUT = K*L

        print(f"\n[Train] Table II DenseNet - max_prod/{pname}")
        if TF:
            nn_mp = build_table2_nn(IN, OUT)
            nn_mp.summary()
            train_model(nn_mp, X_tr_mp, y_tr_mp, X_va_mp, y_va_mp,
                        lr=1e-3, epochs=200, batch=256)
        else:
            nn_mp = NumpyDenseNet([IN,64,32,32,32,5,OUT])
            nn_mp.fit(X_tr_mp, y_tr_mp, X_va_mp, y_va_mp,
                      epochs=300, lr=1e-3, batch=256)

        print(f"\n[Train] Table IV LSTM - max_min/{pname}")
        if TF:
            nn_mm = build_table4_nn(IN, OUT, L, K)
            nn_mm.summary()
            train_model(nn_mm, X_tr_mm, y_tr_mm, X_va_mm, y_va_mm,
                        lr=1e-3, epochs=200, batch=256)
        else:
            nn_mm = NumpyDenseNet([IN,256,128,64,OUT])
            nn_mm.fit(X_tr_mm, y_tr_mm, X_va_mm, y_va_mm,
                      epochs=300, lr=1e-3, batch=256)

        print(f"\n[Eval] {pname} ...")
        res_mp = evaluate(X_mp_te, y_mp_te, nn_mp, precoder,
                          Xm_mp, Xs_mp, use_tf=TF)
        res_mm = evaluate(X_mm_te, y_mm_te, nn_mm, precoder,
                          Xm_mm, Xs_mm, use_tf=TF)

        mse_mp = np.mean((res_mp[0]-res_mp[1])**2)
        mse_mm = np.mean((res_mm[0]-res_mm[1])**2)
        print(f"  Max-Prod avg MSE of SE = {mse_mp:.5f}  "
              f"(paper: MR=0.051, M-MMSE=0.007)")
        print(f"  Max-Min  avg MSE of SE = {mse_mm:.5f}")

        results_fig1[pname] = res_mp
        results_fig2[pname] = res_mm

    print("\n[Plotting] Generating paper figures ...")
    plot_fig1(results_fig1['MR'], results_fig1['M-MMSE'],
              'outputs/paper_Fig1_maxprod_CDF.png')
    plot_fig2(results_fig2['MR'], results_fig2['M-MMSE'],
              'outputs/paper_Fig2_maxmin_CDF.png')

    # =============================================================
    # ACCURACY SUMMARY
    # =============================================================
    print("\n" + "="*60)
    print("  ACCURACY SUMMARY")
    print("="*60)

    for pname in results_fig1:
        se_o, se_n, mse_arr = results_fig1[pname]
        acc     = np.mean(se_n) / (np.mean(se_o) + 1e-10) * 100
        mse_val = np.mean((se_o - se_n)**2)
        p5_o    = np.percentile(se_o, 5)
        p5_n    = np.percentile(se_n, 5)
        print(f"\n  [Max-Prod / {pname}] — Table II DenseNet")
        print(f"    Mean SE Optimal : {np.mean(se_o):.4f} bit/s/Hz")
        print(f"    Mean SE NN      : {np.mean(se_n):.4f} bit/s/Hz")
        print(f"    Accuracy        : {acc:.2f}% of optimal")
        print(f"    5th pct SE      : Optimal={p5_o:.4f}  NN={p5_n:.4f}")
        print(f"    Avg MSE of SE   : {mse_val:.6f}")
        print(f"    (Paper reports  : MR=0.051, M-MMSE=0.007)")

    for pname in results_fig2:
        se_o, se_n, mse_arr = results_fig2[pname]
        acc     = np.mean(se_n) / (np.mean(se_o) + 1e-10) * 100
        mse_val = np.mean((se_o - se_n)**2)
        p5_o    = np.percentile(se_o, 5)
        p5_n    = np.percentile(se_n, 5)
        print(f"\n  [Max-Min / {pname}] — Table IV LSTM")
        print(f"    Mean SE Optimal : {np.mean(se_o):.4f} bit/s/Hz")
        print(f"    Mean SE NN      : {np.mean(se_n):.4f} bit/s/Hz")
        print(f"    Accuracy        : {acc:.2f}% of optimal")
        print(f"    5th pct SE      : Optimal={p5_o:.4f}  NN={p5_n:.4f}")
        print(f"    Avg MSE of SE   : {mse_val:.6f}")

    print("\n" + "="*60)
    print("  DONE.")
    print("  outputs/paper_Fig1_maxprod_CDF.png  ->  Paper Figure 1")
    print("  outputs/paper_Fig2_maxmin_CDF.png   ->  Paper Figure 2")
    print("="*60)
    print("\nExpected results (from paper):")
    print("  Fig 1: M-MMSE NN curve close to optimal, MR has small gap")
    print("         MSE: M-MMSE=0.007, MR=0.051 (paper values)")
    print("  Fig 2: Both MR and M-MMSE NN match optimal closely")

if __name__ == '__main__':
    main()

TensorFlow 2.19.0 found - using Keras (exact paper implementation)
  Reproducing Paper: DL Power Allocation in Massive MIMO
  Sanguinetti, Zappone, Debbah (arXiv:1812.03640)

  Generating datasets for MR

[Dataset] max_prod / MR  N=3000+500
    [max_prod/mr] 0/3000
    [max_prod/mr] 300/3000
    [max_prod/mr] 600/3000
    [max_prod/mr] 900/3000
    [max_prod/mr] 1200/3000
    [max_prod/mr] 1500/3000
    [max_prod/mr] 1800/3000
    [max_prod/mr] 2100/3000
    [max_prod/mr] 2400/3000
    [max_prod/mr] 2700/3000
    [max_prod/mr] 0/500
    [max_prod/mr] 300/500

[Dataset] max_min / MR  N=3000+500
    [max_min/mr] 0/3000
    [max_min/mr] 300/3000
    [max_min/mr] 600/3000
    [max_min/mr] 900/3000
    [max_min/mr] 1200/3000
    [max_min/mr] 1500/3000
    [max_min/mr] 1800/3000
    [max_min/mr] 2100/3000
    [max_min/mr] 2400/3000
    [max_min/mr] 2700/3000
    [max_min/mr] 0/500
    [max_min/mr] 300/500

  Generating datasets for M-MMSE

[Dataset] max_prod / M-MMSE  N=3000+500
    [max_pro

Model: "Table2_DenseNet"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 40)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         2,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 5)              │           165 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 20)             │           120 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,101 (27.74 KB)

 Trainable params: 7,101 (27.74 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 40)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_1 (Reshape)             │ (None, 4, 10)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 4, 256)         │       273,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 128)            │       197,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 20)             │         1,300 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 480,084 (1.83 MB)

 Trainable params: 480,084 (1.83 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step - loss: 0.8180 - val_loss: 0.5094 - learning_rate: 0.0010
Epoch 2/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.4625 - val_loss: 0.4342 - learning_rate: 0.0010
Epoch 3/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.4184 - val_loss: 0.4190 - learning_rate: 0.0010
Epoch 4/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.4096 - val_loss: 0.4131 - learning_rate: 0.0010
Epoch 5/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.4054 - val_loss: 0.4100 - learning_rate: 0.0010
Epoch 6/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.4023 - val_loss: 0.4084 - learning_rate: 0.0010
Epoch 7/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.4011 - val_loss: 0.4110 - learning_rate: 0.0010
Epoch 8/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.3992 - val_loss: 0.4085 - learning_rate: 0.0010
Epoch 9/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.3966 - val_loss: 0.4101 - learning_rate: 0.0010
Epoch 10/2